# Silver layer data transformation of Customer info table
Location: workspace.bronze.crm_cust_info
## Steps:
1. Data types are appropriate.
2. Normalize data for marraige and gender.
3. Rename column headers.


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")
df.count()

## Remove NULL rows for Customer ID

4 records are removed here

In [0]:
df = df.filter(col("cst_id").isNotNull())
df.count()

## Rename column headers

In [0]:
column_headers = {
    'cst_id': 'Customer_ID',
    'cst_key': 'Customer_Key',
    'cst_firstname': 'First_Name',
    'cst_lastname': 'Last_Name',
    'cst_marital_status': 'Marital_Status',
    'cst_gndr': 'Gender',
    'cst_create_date': 'Create_Date'
}

for old, new in column_headers.items():
    df = df.withColumnRenamed(old, new)

df.display()

## Normalize the data categories for gender and marital status

In [0]:
# Gender
df = df.withColumn("Gender", F.when(df.Gender == "M", "Male")
                              .when(df.Gender == "F", "Female")
                              .otherwise("Unknown"))

df = df.withColumn("Marital_Status", F.when(df.Marital_Status == "M", "Married")
                              .when(df.Marital_Status == "S", "Single")
                              .otherwise("Unknown"))
df.display()

## Save the delta table to silver

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("silver.crm_customer")